# The Cellular Interface, Multicellular

_Investigation `the-cellular-interface-multicellular` — coder reproduction notebook._

**Question.** The companion investigation `draft-to-living-cell` specified the paper's nine
interface-composition patterns and ran them with toy-real, lumped mechanisms (ODEs, scripted
events, 1-D or 9-cell grids). This investigation asks whether the same patterns hold when the
environment is not a lumped store but a real 2D spatial field, and the cell is not a scalar
state bundle but a Cellular Potts Model (CPM) agent with genuine shape, footprint, and
motion. Concretely: does the Fig 5 sense/act loop survive being realized as a real CPM cell
(viva-cpm) sitting in a spatio-flux diffusing nutrient field, running dynamic flux-balance
analysis at its own footprint, growing, and secreting a metabolic byproduct back into the
field it depends on — composed from two independently-developed simulation frameworks through
one typed coupling interface (`CpmCellField`)?

This investigation is the spatial sibling of `draft-to-living-cell`. That
investigation turned the cellular-interface paper's nine composition patterns into inert
typed drafts compiled into small, lumped executables (ODE-scale mechanisms on 1-D or 9-cell
grids). This investigation asks the harder question: do those same patterns still compose
when the frameworks on each side of the interface are real, independently-built 2D spatial
simulators — a Cellular Potts Model (viva-cpm, a Rust-backed lattice engine with genuine cell
shape, adhesion, and volume dynamics) on one side, and spatio-flux's diffusion-advection PDE
solver plus dynamic flux-balance analysis on the other? Composing two frameworks that were
never designed with each other in mind is exactly the paper's central claim about typed
interfaces made concrete: `CpmCellField`, the flagship's coupling process, is the interface
where a CPM cell's footprint (a set of lattice pixels) is translated into a spatio-flux
field read, and where a dynamic-FBA solution (mmol/gDW/hr fluxes) is translated back into a
per-pixel field delta and a CPM target-volume update. Four studies are built and run so far —
`cell-environment-coupling-spatial` (the flagship, the spatial analogue of Fig 5's sense/act
loop), `cell-cell-coupling-spatial`, `disintegration-spatial`, and
`growth-and-division-spatial`. The remaining patterns from `draft-to-living-cell` (molecular
interfaces, biomolecular complementarity, autopoiesis, development-and-evolution, and the
cellular-interface contract itself) are named as future increments in the design spec; some
are planned, one (biomolecular complementarity) is in progress with code done but its study
not yet accepted.

The investigations are deliberate analogues, study-for-study, wherever the counterpart
exists: `cell-environment-coupling-spatial` here is the spatial realization of
`draft-to-living-cell/cell-environment-coupling`. Read them side by side — the lumped 9-cell
1-D executable there against the real 60x60 2D CPM+spatio-flux composite here — for the same
interface claim (sensing and acting are one coupling, not two subsystems; the cell measurably
reshapes the gradient it depends on) demonstrated at two very different levels of mechanistic
resolution.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/meta-modelers-guide/meta-modelers-guide').is_dir():
    REPO = Path('/home/runner/work/meta-modelers-guide/meta-modelers-guide')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from meta_modelers_guide.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Cell–Environment Coupling, Spatial (flagship) (`cell-environment-coupling-spatial`)

**Question.** When a single cell metabolizes at its own location, does it measurably reshape the very
nutrient gradient it depends on — niche construction (odling1996) — closing the paper's
§Cell–environment coupling sense → metabolize → secrete loop (Fig 5) as real spatial
metabolism: one cell sensing local glucose, running flux-balance analysis at its own footprint,
growing, and secreting a byproduct that diffuses back into the field it senses? The paper
frames this feedback loop as "the minimal structure required for control"; here the loop is
genuinely closed (a real depletion → uptake feedback), while directed control — chemotaxis up
the sensed gradient — is the deferred half. Mechanistically, the cell and its environment are
two real, independently-built simulators (viva-cpm's Cellular Potts engine, spatio-flux's
diffusion-advection and dynamic-FBA processes) composed through one typed coupling process
(`CpmCellField`), rather than a scripted or lumped stand-in for that coupling.

**Claim.** Composing a real CPM cell with a real spatio-flux nutrient field through one typed
coupling process (`CpmCellField`) reproduces the paper's §Cell–environment coupling
sense → metabolize → secrete loop (Fig 5) as genuine spatial metabolism, not a scripted stand-in
for it (the loop is closed; directed control/chemotaxis is deferred): over 20 ticks the
cell runs dynamic-FBA at its own
footprint (`e_coli_core`, O2-capped to force acetate overflow), grows ~3.4x in CPM volume (32
-> 110 px, ~3% of a 60x60 lattice it does not come close to filling), depletes local glucose by
up to ~18% before diffusion partially resupplies it (niche construction), and secretes a
diffusing acetate plume — two independently-built simulators talking through one typed port.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `single-cell-in-a-field` | `meta_modelers_guide.composites.single-cell-in-a-field` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.single-cell-in-a-field`** — `spec_meta_modelers_guide_composites_single_cell_in_a_field` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_single_cell_in_a_field = load_spec(REPO / 'meta_modelers_guide/composites/single-cell-in-a-field.composite.json')
describe_spec(spec_meta_modelers_guide_composites_single_cell_in_a_field)

In [ ]:
# === Edit parameters for composite 'single-cell-in-a-field' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'diff'  (local:!spatio_flux.processes.diffusion_advection.DiffusionAdvection)
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['n_bins'] = [60, 60]
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['bounds'] = [60.0, 60.0]
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['diffusion_coeffs']['glucose'] = 0.4
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['diffusion_coeffs']['acetate'] = 0.6
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['boundary_conditions']['glucose']['default']['type'] = 'neumann'
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['diff']['config']['boundary_conditions']['acetate']['default']['type'] = 'neumann'

# process 'cell'  (local:CpmCellField)
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['nx'] = 60
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['ny'] = 60
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['seed_block'] = [15, 27, 0, 22, 34, 1]
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['mcs_per_update'] = 8
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['temperature'] = 10.0
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['lambda_volume'] = 2.0
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['contact_j'] = 14.0
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['biomass0'] = 0.1
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['grow_per_biomass'] = 300.0
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['box_volume_L'] = 0.3
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['glucose_km'] = 0.5
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['glucose_vmax'] = 3.5
spec_meta_modelers_guide_composites_single_cell_in_a_field['state']['cell']['config']['oxygen_vmax'] = 2.5

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: cell-environment-coupling-spatial ===
STUDY = 'cell-environment-coupling-spatial'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**single-cell-in-a-field-movie**


In [ ]:
# single-cell-in-a-field-movie
show_viz(_render_one('image:viz/single-cell-in-a-field.gif', {'chart': 'image', 'caption': 'Glucose (with footprint + COM) | glucose depletion halo (Δ from t=0) | acetate\nplume, over the 20-tick flagship run. The middle panel is the direct niche-construction\nsignal: the cell measurably reshapes the very gradient it depends on.'}, RUNS_DB, STUDY_YAML))

**single-cell-in-a-field-metrics**


In [ ]:
# single-cell-in-a-field-metrics
show_viz(_render_one('html:single-cell-in-a-field-metrics.html', {'chart': 'html', 'caption': 'Synced metrics for the flagship run — local_nutrient / biomass / acetate_secreted\n(left axis) against CPM volume (right axis) over 20 ticks.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cell-metabolizes-and-grows | kind=observable path=obs.biomass expr=last(obs.biomass) | op > value 0.1 provenance over 20 ticks, biomass 0.1 -> ~0.37 (tests/test_flagship_field.py::test_flagship_sense_act_loop; tests/test_cpm_cell_field.py::test_cell_metabolizes_grows_and_reshapes_field asserts the same pattern on a smaller 40x40 grid over 12 ticks) |
| cpm-volume-grows | kind=observable path=obs.volume expr=last(obs.volume) | op > value 40.0 provenance CPM volume 32 -> 110 px over 20 ticks (~3.4x), ~3% of the 60x60=3600px lattice (tests/test_flagship_field.py::test_flagship_sense_act_loop) |
| acetate-secreted-and-diffuses | kind=observable path=fields.acetate expr=sum(fields.acetate) | op > value 0.0 provenance field-wide acetate 0 -> ~47.0 over 20 ticks, spreading into a plume around the cell's footprint (tests/test_flagship_field.py::test_flagship_sense_act_loop; the plume is the third panel of the baked GIF) |
| net-glucose-consumed | kind=observable path=fields.glucose expr=sum(fields.glucose) | op < value 5940.0 provenance field-wide glucose total 5940.0 -> ~5908.4 over 20 ticks (~32 units net consumed); the local footprint drawdown is much sharper (up to ~18%) than the field-wide total because only the cell's footprint and its immediate diffusive neighborhood are touched (tests/test_flagship_field.py::test_flagship_sense_act_loop) |
| o2-cap-forces-acetate-overflow | kind=observable path=fields.acetate expr=sum(fields.acetate) with oxygen uncapped vs capped | op approx_equal value 0.0 provenance field-wide acetate over 20 ticks: capped flagship 0 -> ~47.0, O2-uncapped 0 -> 0.0 (exactly), a >100x collapse; uncapped biomass ~2.35 vs capped ~0.37 (the cell respires fully instead of overflowing) (tests/test_flagship_field.py::test_o2_cap_is_what_forces_acetate_overflow; composite meta_modelers_guide/composites/single-cell-in-a-field-o2uncapped.composite.json) |
| field-mass-balance-closes | kind=observable path=fields.glucose, fields.acetate expr=abs(final_mass - (initial_mass + Σ writer_net_exchange)) per species | op < value 1e-06 provenance both glucose and acetate ledgers close to < 1e-6 over 15 ticks; cell net glucose < 0 (sink) and net acetate > 0 (source), diff net ~0 for both (tests/test_mass_balance.py::test_flagship_field_mass_balance_closes; meta_modelers_guide/analysis/mass_balance.py) |


## Study: Cell–Cell Coupling, Spatial (`cell-cell-coupling-spatial`)

**Question.** Does the paper's §Cell–cell coupling viability-negotiation pattern (no dedicated
figure) hold as real spatial multicellularity — multiple CPM cells sharing one diffusing
nutrient field, each running its own dynamic-FBA step at its own footprint, producing both
competitive exclusion and cross-feeding — when the whole colony is composed from independent
frameworks through one N-cell coupling process (`CpmColonyField`), rather than the lumped
two-role, shared-scalar-pool stand-in `draft-to-living-cell`'s §Cell–cell coupling study used?

**Claim.** Composing N real CPM cells with one shared, real spatio-flux nutrient field through a
single N-cell coupling process (`CpmColonyField`) reproduces the paper's §Cell–cell coupling
viability-negotiation pattern (no dedicated figure) as genuine spatial multicellularity, not a
scripted or lumped stand-in for it: over
20-tick runs, an uptake-capacity asymmetry alone (glucose_vmax 10 vs 4) drives asymmetric
growth toward competitive exclusion (biomass 237.9 vs 64.5, a 3.69x margin; volume 3511 vs 81
px — the loser displaced but still viable, above its 1.25 seed, not driven below a viability
bound), while a
different composite over the same coupling process — a localized glucose depot, a fast-diffusing
acetate byproduct, and an acetate-consuming role — sustains a cell that never touches glucose
(local glucose 0.0) purely via a diffusing cross-feeding handoff (local acetate 1.30, biomass
1.25 -> 3.79). Both regimes run through the same process; only the composite's cell roles and
field constants differ, mirroring `draft-to-living-cell/cell-cell-coupling`'s law-4 pattern
(two handler environments negotiate the shared-resource constraint differently over one
interface) at real spatial, per-footprint metabolic resolution.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `cellcell-compete` | `meta_modelers_guide.composites.cellcell-compete` | 0 | — |
| `cellcell-crossfeed` | `meta_modelers_guide.composites.cellcell-crossfeed` | 0 | — |
| `cellcell-crossfeed-knockout` | `meta_modelers_guide.composites.cellcell-crossfeed-knockout` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.cellcell-compete`** — `spec_meta_modelers_guide_composites_cellcell_compete` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_cellcell_compete = load_spec(REPO / 'meta_modelers_guide/composites/cellcell-compete.composite.json')
describe_spec(spec_meta_modelers_guide_composites_cellcell_compete)

In [ ]:
# === Edit parameters for composite 'cellcell-compete' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'diff'  (local:!spatio_flux.processes.diffusion_advection.DiffusionAdvection)
spec_meta_modelers_guide_composites_cellcell_compete['state']['diff']['config']['n_bins'] = [60, 60]
spec_meta_modelers_guide_composites_cellcell_compete['state']['diff']['config']['bounds'] = [60.0, 60.0]
spec_meta_modelers_guide_composites_cellcell_compete['state']['diff']['config']['diffusion_coeffs']['glucose'] = 0.4
spec_meta_modelers_guide_composites_cellcell_compete['state']['diff']['config']['diffusion_coeffs']['acetate'] = 0.6
spec_meta_modelers_guide_composites_cellcell_compete['state']['diff']['config']['boundary_conditions']['glucose']['default']['type'] = 'neumann'
spec_meta_modelers_guide_composites_cellcell_compete['state']['diff']['config']['boundary_conditions']['acetate']['default']['type'] = 'neumann'

# process 'colony'  (local:CpmColonyField)
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['grid']['nx'] = 60
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['grid']['ny'] = 60
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['mcs'] = 3
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['box_volume_L'] = 0.3
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['grow_per_biomass'] = 40.0
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][0]['role'] = 'competitor'
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][0]['seed_block'] = [12, 26, 0, 20, 34, 1]
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][0]['glucose_vmax'] = 10.0
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][0]['oxygen_vmax'] = 15.0
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][0]['target_volume'] = 50.0
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][0]['lambda_volume'] = 2.0
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][1]['role'] = 'competitor'
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][1]['seed_block'] = [40, 26, 0, 48, 34, 1]
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][1]['glucose_vmax'] = 4.0
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][1]['oxygen_vmax'] = 15.0
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][1]['target_volume'] = 50.0
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['cells'][1]['lambda_volume'] = 2.0
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['contact'][0]['a'] = 0
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['contact'][0]['b'] = 1
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['contact'][0]['j'] = 14.0
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['contact'][1]['a'] = 1
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['contact'][1]['b'] = 1
spec_meta_modelers_guide_composites_cellcell_compete['state']['colony']['config']['contact'][1]['j'] = 14.0

**Composite `meta_modelers_guide.composites.cellcell-crossfeed`** — `spec_meta_modelers_guide_composites_cellcell_crossfeed` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_cellcell_crossfeed = load_spec(REPO / 'meta_modelers_guide/composites/cellcell-crossfeed.composite.json')
describe_spec(spec_meta_modelers_guide_composites_cellcell_crossfeed)

In [ ]:
# === Edit parameters for composite 'cellcell-crossfeed' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'diff'  (local:!spatio_flux.processes.diffusion_advection.DiffusionAdvection)
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['diff']['config']['n_bins'] = [60, 60]
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['diff']['config']['bounds'] = [60.0, 60.0]
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['diff']['config']['diffusion_coeffs']['glucose'] = 0.4
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['diff']['config']['diffusion_coeffs']['acetate'] = 15.0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['diff']['config']['boundary_conditions']['glucose']['default']['type'] = 'neumann'
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['diff']['config']['boundary_conditions']['acetate']['default']['type'] = 'neumann'

# process 'colony'  (local:CpmColonyField)
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['grid']['nx'] = 60
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['grid']['ny'] = 60
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['mcs'] = 3
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['box_volume_L'] = 0.3
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['grow_per_biomass'] = 30.0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][0]['role'] = 'secretor'
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][0]['seed_block'] = [8, 26, 0, 16, 34, 1]
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][0]['glucose_vmax'] = 10.0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][0]['oxygen_vmax'] = 5.0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][0]['target_volume'] = 50.0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][0]['lambda_volume'] = 2.0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][1]['role'] = 'consumer'
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][1]['seed_block'] = [22, 26, 0, 30, 34, 1]
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][1]['acetate_vmax'] = 20.0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][1]['oxygen_vmax'] = 20.0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][1]['target_volume'] = 50.0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['cells'][1]['lambda_volume'] = 2.0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['contact'][0]['a'] = 0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['contact'][0]['b'] = 1
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['contact'][0]['j'] = 14.0
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['contact'][1]['a'] = 1
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['contact'][1]['b'] = 1
spec_meta_modelers_guide_composites_cellcell_crossfeed['state']['colony']['config']['contact'][1]['j'] = 14.0

**Composite `meta_modelers_guide.composites.cellcell-crossfeed-knockout`** — `spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout = load_spec(REPO / 'meta_modelers_guide/composites/cellcell-crossfeed-knockout.composite.json')
describe_spec(spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout)

In [ ]:
# === Edit parameters for composite 'cellcell-crossfeed-knockout' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'diff'  (local:!spatio_flux.processes.diffusion_advection.DiffusionAdvection)
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['diff']['config']['n_bins'] = [60, 60]
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['diff']['config']['bounds'] = [60.0, 60.0]
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['diff']['config']['diffusion_coeffs']['glucose'] = 0.4
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['diff']['config']['diffusion_coeffs']['acetate'] = 15.0
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['diff']['config']['boundary_conditions']['glucose']['default']['type'] = 'neumann'
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['diff']['config']['boundary_conditions']['acetate']['default']['type'] = 'neumann'

# process 'colony'  (local:CpmColonyField)
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['grid']['nx'] = 60
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['grid']['ny'] = 60
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['mcs'] = 3
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['box_volume_L'] = 0.3
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['grow_per_biomass'] = 30.0
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['cells'][0]['role'] = 'consumer'
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['cells'][0]['seed_block'] = [22, 26, 0, 30, 34, 1]
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['cells'][0]['acetate_vmax'] = 20.0
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['cells'][0]['oxygen_vmax'] = 20.0
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['cells'][0]['target_volume'] = 50.0
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['cells'][0]['lambda_volume'] = 2.0
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['contact'][0]['a'] = 0
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['contact'][0]['b'] = 1
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['contact'][0]['j'] = 14.0
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['contact'][1]['a'] = 1
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['contact'][1]['b'] = 1
spec_meta_modelers_guide_composites_cellcell_crossfeed_knockout['state']['colony']['config']['contact'][1]['j'] = 14.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: cell-cell-coupling-spatial ===
STUDY = 'cell-cell-coupling-spatial'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**cellcell-compete-movie**


In [ ]:
# cellcell-compete-movie
show_viz(_render_one('image:viz/cellcell-compete.gif', {'chart': 'image', 'caption': 'Spatial competitive exclusion over 20 ticks -- the higher-vmax competitor (id 1) claims the shared lattice (filled, over the glucose depletion it drives); the lower-vmax competitor (id 2) is pushed to the margins.'}, RUNS_DB, STUDY_YAML))

**cellcell-compete-metrics**


In [ ]:
# cellcell-compete-metrics
show_viz(_render_one('html:viz/cellcell-compete-metrics.html', {'chart': 'html', 'caption': "Biomass divergence for the competition run -- the id 1 / id 2 biomass ratio climbs to a 3.69x final margin over 20 ticks, the study's headline number made directly visible."}, RUNS_DB, STUDY_YAML))

**cellcell-crossfeed-movie**


In [ ]:
# cellcell-crossfeed-movie
show_viz(_render_one('image:viz/cellcell-crossfeed.gif', {'chart': 'image', 'caption': "Spatial cross-feeding over 20 ticks -- the secretor's (id 1) acetate plume reaches the consumer (id 2), which grows despite never touching glucose."}, RUNS_DB, STUDY_YAML))

**cellcell-crossfeed-metrics**


In [ ]:
# cellcell-crossfeed-metrics
show_viz(_render_one('html:viz/cellcell-crossfeed-metrics.html', {'chart': 'html', 'caption': "Synced per-cell metrics for the cross-feeding run -- the consumer's local_glucose stays at 0.0 while local_acetate and biomass rise from the diffused plume, over 20 ticks."}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| competition-excludes-the-slower-cell | kind=observable path=obs.biomass expr=last(obs.biomass["1"]) / last(obs.biomass["2"]) | op > value 1.5 provenance biomass 237.9 (id 1, vmax 10) vs 64.5 (id 2, vmax 4) -> 3.69x ratio over 20 ticks (tests/test_cellcell_regimes.py::test_competition_excludes_the_slower_cell, tests/test_cpm_colony_field.py::test_two_cells_metabolize_grow_and_deplete_disjointly) |
| competition-volume-tracks-biomass | kind=observable path=obs.volume expr=last(obs.volume["1"]) > last(obs.volume["2"]) | op > value 0.0 provenance volume 3511 px (id 1) vs 81 px (id 2) over 20 ticks -- the faster competitor claims most of the shared lattice (tests/test_cellcell_regimes.py::test_competition_excludes_the_slower_cell) |
| competition-loser-displaced-not-excluded | kind=observable path=obs.biomass expr=last(obs.biomass["2"])   # loser final biomass, vs the 1.25 viable-biomass floor | op > value 1.25 provenance loser (id 2) ends at biomass 64.5 -- it GREW ~51.6x from its 1.25 seed and sits far above the viable-biomass floor, while holding only 81 px against the winner's 3511 px (~97.5% of the 3600-px lattice): spatially displaced, not excluded. Extending the run to t≈30 fills the lattice and crowds the still-viable loser off it (eviction, not starvation) (tests/test_cellcell_regimes.py::test_competition_loser_is_displaced_not_excluded) |
| crossfeeding-consumer-blind-to-glucose | kind=observable path=obs.local_glucose expr=last(obs.local_glucose["2"]) | op < value 0.5 provenance consumer (id 2) local_glucose == 0.0 over 20 ticks (tests/test_cellcell_regimes.py::test_crossfeeding_keeps_the_consumer_viable) |
| crossfeeding-consumer-grows-on-acetate-alone | kind=observable path=obs.biomass expr=last(obs.biomass["2"]) | op > value 1.25 provenance consumer (id 2) biomass 1.25 -> 3.79 (roughly 3x, back-loaded to the last ~5 ticks once the acetate plume arrives) over 20 ticks (tests/test_cellcell_regimes.py::test_crossfeeding_keeps_the_consumer_viable) |
| crossfeeding-acetate-plume-reaches-consumer | kind=observable path=obs.local_acetate expr=last(obs.local_acetate["2"]) | op > value 0.0 provenance consumer (id 2) local_acetate == 1.30 by t=20, confirming the fast-diffusing (coefficient 15.0 vs glucose's 0.4) acetate plume reaches the consumer's footprint (tests/test_cellcell_regimes.py::test_crossfeeding_keeps_the_consumer_viable) |
| two-writer-additivity-conserves-mass | kind=observable path=fields.glucose expr=net change in sum(fields.glucose) over 3 ticks (final minus initial) | op == value -6.0 provenance a synthetic structural-control process (not this study's composites): two single-pixel writers on disjoint pixels, each removing 1.0/tick over 3 ticks, sum to a net field-wide change of exactly -6.0 with both pixels independently correct (tests/test_cpm_colony_two_writer.py::test_two_disjoint_writers_sum_and_conserve) |
| crossfeeding-secretor-knockout-consumer-fails-to-grow | kind=observable path=obs.biomass expr=last(obs.biomass["1"])   # consumer-alone final biomass; expected to stay at the seed | op <= value 1.2500001 provenance consumer (id 1) biomass stays at its 1.25 seed and local_acetate stays 0.0 over 20 ticks with the secretor removed -- it persists on the lattice (~35 px) but flatly fails to grow, versus 1.25 -> 3.79 in the full cross-feed regime (tests/test_cellcell_regimes.py::test_crossfeeding_knockout_consumer_fails_to_grow) |


## Study: Disintegration, Spatial (`disintegration-spatial`)

**Question.** Does the paper's §Disintegration level-shift (Fig 6) hold spatially -- a coherent CPM cell that loses structural
viability when a diffusing field of acetate (the SAME metabolic byproduct that is waste-and-food
in `cell-cell-coupling-spatial`, here toxic at high local concentration) crosses its viability
bound, its domain resorbing while its shed material becomes scattering physical particles -- composed from independent frameworks
(viva-cpm + spatio-flux particles) through one coupling process (`CpmDisintegration` +
`BrownianMovement`)?

**Claim.** Composing one real CPM cell with a real spatio-flux acetate field and a real spatio-flux
particle-movement process through two coupling processes (`CpmDisintegration` for the
viability-collapse trigger and resorption, stock `BrownianMovement` for the debris scatter)
reproduces the paper's §Disintegration level-shift (Fig 6) as genuine spatial dissolution, not a scripted or lumped
stand-in for it: over a fixed-seed deterministic 20-tick CPM run, the cell holds coherent
(area > 30) through tick 6, the footprint-local acetate mean crosses viability_threshold at
released_tick = 7, the domain resorbs to area 0 by tick 16, and its 68 shed pixels become a
debris cloud whose RMS spread strictly increases through tick 24 (the trajectory -- hold, cross,
resorb, particle count -- reproduces exactly; the scatter cloud moves under unseeded
`BrownianMovement`, so its exact RMS magnitudes are representative of one run, not
run-to-run-reproducible -- only the strict-increase direction is the robust claim) -- hold,
cross, resorb, scatter, end to end from one coupling process plus one stock movement process,
mirroring `draft-to-living-cell/disintegration`'s §Disintegration level-shift pattern (Fig 6) at
real spatial, per-pixel resolution.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `disintegration-spatial` | `meta_modelers_guide.composites.disintegration-spatial` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.disintegration-spatial`** — `spec_meta_modelers_guide_composites_disintegration_spatial` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_disintegration_spatial = load_spec(REPO / 'meta_modelers_guide/composites/disintegration-spatial.composite.json')
describe_spec(spec_meta_modelers_guide_composites_disintegration_spatial)

In [ ]:
# === Edit parameters for composite 'disintegration-spatial' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'diff'  (local:!spatio_flux.processes.diffusion_advection.DiffusionAdvection)
spec_meta_modelers_guide_composites_disintegration_spatial['state']['diff']['config']['n_bins'] = [60, 60]
spec_meta_modelers_guide_composites_disintegration_spatial['state']['diff']['config']['bounds'] = [60.0, 60.0]
spec_meta_modelers_guide_composites_disintegration_spatial['state']['diff']['config']['diffusion_coeffs']['acetate'] = 4.0
spec_meta_modelers_guide_composites_disintegration_spatial['state']['diff']['config']['boundary_conditions']['acetate']['default']['type'] = 'neumann'

# process 'cell'  (local:CpmDisintegration)
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['grid']['nx'] = 60
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['grid']['ny'] = 60
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['bounds']['x'] = 60.0
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['bounds']['y'] = 60.0
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['cell']['seed_block'] = [26, 26, 0, 34, 34, 1]
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['cell']['target_volume'] = 64.0
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['cell']['lambda_volume'] = 2.0
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['cell']['temperature'] = 11.0
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['contact'][0]['a'] = 0
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['contact'][0]['b'] = 1
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['contact'][0]['j'] = 14.0
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['viability_threshold'] = 0.5
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['resorb_per_tick'] = 6.0
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['max_particles_per_tick'] = 8
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['mcs'] = 3
spec_meta_modelers_guide_composites_disintegration_spatial['state']['cell']['config']['stressor_field'] = 'acetate'

# process 'move'  (local:!spatio_flux.processes.particles.BrownianMovement)
spec_meta_modelers_guide_composites_disintegration_spatial['state']['move']['config']['bounds'] = [60.0, 60.0]
spec_meta_modelers_guide_composites_disintegration_spatial['state']['move']['config']['n_bins'] = [60, 60]
spec_meta_modelers_guide_composites_disintegration_spatial['state']['move']['config']['diffusion_rate'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: disintegration-spatial ===
STUDY = 'disintegration-spatial'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**disintegration-spatial-movie**


In [ ]:
# disintegration-spatial-movie
show_viz(_render_one('image:viz/disintegration-spatial.gif', {'chart': 'image', 'caption': 'Spatial disintegration -- the cell holds, then resorbs once the diffusing acetate field crosses its viability bound at tick 7, shedding a scattering debris cloud.'}, RUNS_DB, STUDY_YAML))

**disintegration-metrics**


In [ ]:
# disintegration-metrics
show_viz(_render_one('html:viz/disintegration-metrics.html', {'chart': 'html', 'caption': 'Synced metrics for the disintegration run -- area collapses at released_tick == 7, particle count climbs to 68 and holds, over 20-24 ticks.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cell-holds-coherent-before-crossing | kind=observable path=obs.area expr=obs.area at tick 6 | op > value 30.0 provenance area 61-66 through tick 6, released stays False, n_particles == 0 (tests/test_disintegration_regime.py::test_cell_holds_then_disintegrates_into_scattering_debris, tests/test_cpm_disintegration.py::test_holds_below_threshold) |
| stressor-crosses-viability-bound | kind=observable path=obs.released_tick expr=obs.released_tick once obs.released first reads True | op == value 7.0 provenance released latches True at released_tick == 7.0, deterministic across repeated runs of the same composite (tests/test_cpm_disintegration.py::test_releases_and_disintegrates_into_particles, tests/test_disintegration_regime.py) |
| domain-resorbs-to-zero | kind=observable path=obs.area expr=obs.area at tick 16 | op == value 0.0 provenance area falls monotonically from 56 (tick 7) to 0 by tick 16, and stays 0 through tick 20 (tests/test_disintegration_regime.py::test_cell_holds_then_disintegrates_into_scattering_debris, tests/test_cpm_disintegration.py::test_releases_and_disintegrates_into_particles) |
| shed-material-becomes-debris-particles | kind=observable path=particles expr=len(particles) once area reaches 0 | op >= value 20.0 provenance exactly 68 debris particles shed by tick 16 (8/tick capped, deterministic across repeated runs), staying at 68 for the remainder of the run (tests/test_disintegration_regime.py::test_cell_holds_then_disintegrates_into_scattering_debris) |
| debris-cloud-keeps-scattering | kind=observable path=particles expr=RMS radius of particle positions, tick 22 vs tick 16 | op > value 0.0 provenance RMS spread grows monotonically over an 8-tick post-dissolution window from ~5.3 (tick 16, dissolution complete) to ~7.6 (tick 24). BrownianMovement is unseeded, so the exact magnitudes vary run-to-run (representative values shown); the strict INCREASE is the robust, test-asserted claim (tests/test_disintegration_regime.py::test_cell_holds_then_disintegrates_into_scattering_debris) |
| particle-bridge-emits-then-scatters | kind=observable path=particles expr=len(particles) unchanged and >=2 positions moved, tick 1 vs tick 4 | op == value 3.0 provenance a synthetic emitter process (not this study's composite): 3 particles emitted on tick 1, still exactly 3 particles at tick 4, with at least 2 having moved (tests/test_particle_bridge_spike.py::test_emit_then_scatter) |


## Study: Growth and Division, Spatial (`growth-and-division-spatial`)

**Question.** A cell grows, crosses a threshold, and divides -- and its daughters do not necessarily divide together. This is the paper's §Growth and division growth-then-divide pattern (Fig 10a,b): metabolism drives a cell's own state across a threshold, crossing it fires division and partitions state across the two resulting daughters, and those daughters, "remain[ing] coupled through shared environmental state," can either stay in step or diverge (§Growth and division).
Does this hold spatially -- a single CPM cell whose metabolism (dFBA solved at its own lattice footprint) grows its volume until it crosses `vol_threshold` and divides via the native CPM engine operation `divide_cells`, partitioning biomass mass-conserved across the two daughters, compounding into a small multi-generation lineage tree -- and do sibling cells sharing one glucose field begin growing at different local rates as the colony crowds, so their division timing desynchronizes rather than staying locked to clean powers of two? Composed from independent frameworks (viva-cpm + spatio-flux + cobra) through one coupling process (`CpmGrowthDivision`).

**Claim.** Metabolism-driven growth crossing a threshold fires division, and division partitions state -- biomass, and lattice pixels -- across two daughters that resume growing themselves, compounding into a genuine multi-generation lineage tree whose siblings desynchronize as they compete for shared, local resources: this is the paper's §Growth and division growth-then-divide pattern (Fig 10a,b), including its "coordination and divergence" clause, realized spatially. One real CPM cell, growing via real per-footprint dFBA metabolism, divides via the native CPM engine operation `divide_cells` (biomass partitioned proportional to post-split daughter volume, not reset and discarded) through a single coupling process (`CpmGrowthDivision`). Over a 36-tick run sampled every 3 ticks: the founder compounds into an 18-cell lineage tree spanning 4 generations (`max_generation` = 4), the population steps up in a monotonic non-decreasing staircase (1,2,2,2,4,4,5,8,8,11,14,18) that departs from clean powers of two from tick 21 onward as sibling cells' division timing diverges, and every cell's CPM volume stays bounded in a sawtooth between `reset_target` = 40 and `vol_threshold` = 80. The staircase is bounded more by lattice crowding (the colony packs to roughly two-thirds occupancy within its own footprint) than by nutrient limitation (field-wide glucose depletes only ~3% over the run). This is direct simulation output from three independently-developed frameworks (viva-cpm + spatio-flux + cobra) meeting at one typed interface. It mirrors `draft-to-living-cell/growth-and-division`'s §Growth and division claim (Fig 10a,b) (growth drives the cell's own state across a threshold; crossing the threshold triggers division) but realizes division as the native spatial CPM operation `divide_cells` acting directly on lattice pixels, in contrast to that study's place-graph rewrite (one node becoming two structurally-new nodes at a lumped, non-spatial level of composition).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `growth-division-spatial` | `meta_modelers_guide.composites.growth-division-spatial` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.growth-division-spatial`** — `spec_meta_modelers_guide_composites_growth_division_spatial` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_growth_division_spatial = load_spec(REPO / 'meta_modelers_guide/composites/growth-division-spatial.composite.json')
describe_spec(spec_meta_modelers_guide_composites_growth_division_spatial)

In [ ]:
# === Edit parameters for composite 'growth-division-spatial' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'diff'  (local:!spatio_flux.processes.diffusion_advection.DiffusionAdvection)
spec_meta_modelers_guide_composites_growth_division_spatial['state']['diff']['config']['n_bins'] = [60, 60]
spec_meta_modelers_guide_composites_growth_division_spatial['state']['diff']['config']['bounds'] = [60.0, 60.0]
spec_meta_modelers_guide_composites_growth_division_spatial['state']['diff']['config']['diffusion_coeffs']['glucose'] = 0.4
spec_meta_modelers_guide_composites_growth_division_spatial['state']['diff']['config']['diffusion_coeffs']['acetate'] = 0.6
spec_meta_modelers_guide_composites_growth_division_spatial['state']['diff']['config']['boundary_conditions']['glucose']['default']['type'] = 'neumann'
spec_meta_modelers_guide_composites_growth_division_spatial['state']['diff']['config']['boundary_conditions']['acetate']['default']['type'] = 'neumann'

# process 'cell'  (local:CpmGrowthDivision)
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['grid']['nx'] = 60
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['grid']['ny'] = 60
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['cell']['seed_block'] = [27, 27, 0, 33, 33, 1]
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['cell']['target_volume'] = 40.0
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['cell']['lambda_volume'] = 2.0
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['cell']['temperature'] = 11.0
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['contact'][0]['a'] = 0
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['contact'][0]['b'] = 1
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['contact'][0]['j'] = 14.0
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['contact'][1]['a'] = 1
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['contact'][1]['b'] = 1
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['contact'][1]['j'] = 14.0
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['mcs'] = 3
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['init_biomass'] = 1.25
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['grow_per_biomass'] = 40.0
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['box_volume_L'] = 0.3
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['glucose_km'] = 0.5
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['glucose_vmax'] = 1.5
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['oxygen_vmax'] = 15.0
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['vol_threshold'] = 80.0
spec_meta_modelers_guide_composites_growth_division_spatial['state']['cell']['config']['reset_target'] = 40.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: growth-and-division-spatial ===
STUDY = 'growth-and-division-spatial'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**growth-division-spatial-movie**


In [ ]:
# growth-division-spatial-movie
show_viz(_render_one('image:viz/growth-division-spatial.gif', {'chart': 'image', 'caption': 'Spatial growth-and-division over 36 ticks -- a single founder cell compounds into a lineage of distinguishably lineage/generation-colored daughters (18 cells by t=36 in the re-baked run) as metabolism-driven volume growth repeatedly crosses vol_threshold and native divide_cells splits each cell.'}, RUNS_DB, STUDY_YAML))

**growth-division-metrics**


In [ ]:
# growth-division-metrics
show_viz(_render_one('html:viz/growth-division-metrics.html', {'chart': 'html', 'caption': 'Synced n_cells/total_volume metrics for the growth-division run -- a monotonic non-decreasing population staircase, division events marked, over 36 ticks.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| population-steps-up-by-division | kind=observable path=obs.n_cells expr=max(obs.n_cells) over the 36-tick run | op >= value 8.0 provenance n_cells trajectory 1,2,2,2,4,4,5,8,8,11,14,18 over 36 ticks (12 samples, cadence 3) -- steps 1 -> 2 -> 4 -> 8 -> 18 (tests/test_growth_division_regime.py::test_population_steps_up_by_division, tests/test_growth_division_viz.py::test_growth_division_gif_and_metrics) |
| population-never-shrinks | kind=observable path=obs.n_cells expr=obs.n_cells == sorted(obs.n_cells) over the 36-tick run | op > value 0.0 provenance n_cells trajectory 1,2,2,2,4,4,5,8,8,11,14,18 is monotonic non-decreasing over the full 36-tick run (tests/test_growth_division_regime.py::test_population_steps_up_by_division asserts ns == sorted(ns)) |
| daughters-resume-growth-without-runaway | kind=observable path=obs.n_cells expr=obs.n_cells after run(30) from a single seed cell (n0 = 1) | op >= value 3.0 provenance starting from n0 = 1, n_cells >= 3 after 30 ticks (at least 1 -> 2 -> ~4), with every per-cell volume bounded in (5, 200) -- no runaway single cell, no zero-volume phantom daughters (tests/test_cpm_growth_division.py::test_cell_grows_and_divides_into_a_population) |
| per-cell-volume-stays-bounded | kind=observable path=obs.volume expr=range of per-cell CPM volume across all cells and all sampled ticks | op < value 200.0 provenance per-cell volumes stay bounded roughly 31-79 px across the 36-tick run (division halves a cell once it crosses vol_threshold = 80, resetting toward reset_target = 40, then it regrows); the tests' hard bounds (8 < v < 200) hold at every sampled tick (tests/test_growth_division_regime.py::test_population_steps_up_by_division, tests/test_cpm_growth_division.py::test_cell_grows_and_divides_into_a_population) |
| native-division-conserves-mass | kind=observable path=cell_volumes expr=(vol_parent + vol_new_daughter) - vol_before_split, and divide_cells(500.0, 40.0) on a sub-threshold cell | op <= value 2.0 provenance divide_cells(80.0, 40.0) on a cell grown to ~150 splits it into two daughters (parent id kept + one new id) whose combined volume matches the pre-split volume within +-2 px (rounding); divide_cells(500.0, 40.0) on a cell at 60 (below threshold) returns [] -- a correct no-op (tests/test_cpm_divide_spike.py::test_divide_splits_one_into_two_mass_conserved, test_below_threshold_is_noop) |
| division-desynchronizes-from-lockstep | kind=observable path=obs.n_cells, obs.generation expr=count of n_cells samples not in {1,2,4,8,16,32}, and number of distinct generation values present among live cells at the final sampled tick | op >= value 1.0 provenance n_cells trajectory 1,2,2,2,4,4,5,8,8,11,14,18 departs from powers of two starting at tick 21 (5 cells) and stays desynchronized through 11, 14, 18; at the final tick, live cells span 5 distinct generations (0 through 4) simultaneously in the same 18-cell colony (tests/test_growth_division_regime.py::test_division_desynchronizes_from_lockstep_powers_of_two) |
| lineage-compounds-into-a-tree | kind=observable path=obs.max_generation expr=max(obs.max_generation) over the 36-tick run | op >= value 3.0 provenance max_generation reaches 4 by the end of the 36-tick run (trajectory 0,1,1,1,2,2,2,3,3,3,3,4 across the 12 sampled ticks); founder id 1 stays generation 0 throughout; every daughter's generation == its recorded parent's generation + 1 (tests/test_growth_division_regime.py::test_lineage_compounds_into_a_multigeneration_tree) |


## Study: Biomolecular Complementarity, Spatial (`biomolecular-complementarity-spatial`)

**Question.** The paper's §Molecular compositions section (Fig 8) frames its subject as a question about interfaces, not architecture: which patterns of molecular complementarity give rise to interfaces that behave as functional, regulatable boundaries, and which produce transient or unregulated aggregates? This study puts two spatial primitives of that question on the lattice. Differential-adhesion cell sorting -- Steinberg's Differential Adhesion Hypothesis -- is complementarity made spatial: a mixed population of two cell types, distinguished only by which contacts they find energetically favorable, demixes so that like sits with like and the heterotypic boundary between them collapses, a direct spatial reading of which interface variables must align for two cells to stay coupled. Biomolecular-condensate phase separation is the diffuse-interface counterpart: a scalar composition field, seeded near-critical, spontaneously separates into two phases divided by a sharp interface -- a boundary forming out of a well-mixed medium, the way a condensate droplet does. Each is a real 2D spatial mechanism run through the process-bigraph engine; the two are independent demonstrations, not one coupled model.

**Claim.** Differential-adhesion cell sorting reproduces the paper's §Molecular compositions complementarity (Fig 8) as a spatial mechanism: two cell types differing only in which contacts they find energetically favorable demix a well-mixed checkerboard (heterotypic-interface fraction ~0.64 -> ~0.06 over ~600 Monte Carlo steps) while staying cohesive (live-cell pixels ~1530, <1% drift; per-type counts constant at 32/32) -- like sitting with like is the spatial reading of which interface variables must align for coupling. Independently, a Cahn-Hilliard composition field realizes the condensate mechanism the paper draws from banani2017: a near-critical field phase-separates into two domains bounded by a sharp, emergent interface (variance ~7e-5 -> ~0.38 over 10000 steps) with mass conserved to ~4e-18 and the field bounded in [-1, 1]. The honest boundary of the claim is explicit: the condensate forms an interface but does not yet do interface work -- it does not selectively concentrate or exclude a second species -- so it demonstrates a boundary forming, not yet the functional, regulatable boundary the paper's central question distinguishes from a transient aggregate. Neither regime computes a binding affinity; both are tuned demonstration primitives, not fitted to a measured tissue or condensate. The two run as independent processes through the process-bigraph engine, with no coupling between them.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `cell-sorting-spatial` | `meta_modelers_guide.composites.cell-sorting-spatial` | 0 | — |
| `condensate-cahn-hilliard` | `meta_modelers_guide.composites.condensate-cahn-hilliard` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.cell-sorting-spatial`** — `spec_meta_modelers_guide_composites_cell_sorting_spatial` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_cell_sorting_spatial = load_spec(REPO / 'meta_modelers_guide/composites/cell-sorting-spatial.composite.json')
describe_spec(spec_meta_modelers_guide_composites_cell_sorting_spatial)

In [ ]:
# === Edit parameters for composite 'cell-sorting-spatial' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'cell'  (local:CpmSorting)
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['grid']['nx'] = 70
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['grid']['ny'] = 70
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['checkerboard']['n'] = 8
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['checkerboard']['size'] = 5
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['checkerboard']['x0'] = 15
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['checkerboard']['y0'] = 15
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][0]['a'] = 0
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][0]['b'] = 1
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][0]['j'] = 8.0
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][1]['a'] = 0
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][1]['b'] = 2
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][1]['j'] = 8.0
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][2]['a'] = 1
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][2]['b'] = 1
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][2]['j'] = 2.0
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][3]['a'] = 2
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][3]['b'] = 2
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][3]['j'] = 2.0
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][4]['a'] = 1
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][4]['b'] = 2
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['contact'][4]['j'] = 11.0
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['temperature'] = 10.0
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['target_volume'] = 25.0
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['lambda_volume'] = 2.0
spec_meta_modelers_guide_composites_cell_sorting_spatial['state']['cell']['config']['mcs'] = 10

**Composite `meta_modelers_guide.composites.condensate-cahn-hilliard`** — `spec_meta_modelers_guide_composites_condensate_cahn_hilliard` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_condensate_cahn_hilliard = load_spec(REPO / 'meta_modelers_guide/composites/condensate-cahn-hilliard.composite.json')
describe_spec(spec_meta_modelers_guide_composites_condensate_cahn_hilliard)

In [ ]:
# === Edit parameters for composite 'condensate-cahn-hilliard' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'ch'  (local:CahnHilliard)
spec_meta_modelers_guide_composites_condensate_cahn_hilliard['state']['ch']['config']['grid']['nx'] = 64
spec_meta_modelers_guide_composites_condensate_cahn_hilliard['state']['ch']['config']['grid']['ny'] = 64
spec_meta_modelers_guide_composites_condensate_cahn_hilliard['state']['ch']['config']['M'] = 1.0
spec_meta_modelers_guide_composites_condensate_cahn_hilliard['state']['ch']['config']['kappa'] = 0.5
spec_meta_modelers_guide_composites_condensate_cahn_hilliard['state']['ch']['config']['dt'] = 0.002
spec_meta_modelers_guide_composites_condensate_cahn_hilliard['state']['ch']['config']['steps_per_tick'] = 200

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: biomolecular-complementarity-spatial ===
STUDY = 'biomolecular-complementarity-spatial'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**cell-sorting-spatial-movie**


In [ ]:
# cell-sorting-spatial-movie
show_viz(_render_one('image:viz/cell-sorting-spatial.gif', {'chart': 'image', 'caption': 'Differential-adhesion cell sorting over ~600 MCS -- a mixed checkerboard demixes so like sits with like, the heterotypic interface collapsing while the clump stays cohesive.'}, RUNS_DB, STUDY_YAML))

**cell-sorting-metrics**


In [ ]:
# cell-sorting-metrics
show_viz(_render_one('html:viz/cell-sorting-metrics.html', {'chart': 'html', 'caption': 'Sorting metrics -- heterotypic-interface fraction collapses (~0.64 -> ~0.06) while the cohesion guard (cell pixels ~1530) holds, so the demixing is real sorting, not dissolution.'}, RUNS_DB, STUDY_YAML))

**condensate-cahn-hilliard-movie**


In [ ]:
# condensate-cahn-hilliard-movie
show_viz(_render_one('image:viz/condensate-cahn-hilliard.gif', {'chart': 'image', 'caption': 'Cahn-Hilliard condensate phase separation over 10000 steps -- a near-critical field separates into two phases bounded by a sharp interface, with mass conserved.'}, RUNS_DB, STUDY_YAML))

**condensate-metrics**


In [ ]:
# condensate-metrics
show_viz(_render_one('html:viz/condensate-metrics.html', {'chart': 'html', 'caption': 'Condensate phase-separation variance -- phi_var climbs from ~7e-5 to ~0.38 over 10000 steps as the field separates into two domains, mass conserved.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sorting-heterotypic-interface-collapses | kind=observable path=obs.hetero_frac expr=last(obs.hetero_frac) | op < value 0.2 provenance hetero_frac ~0.64 at first observation -> ~0.06 by ~600 MCS (raw seeded checkerboard ~1.0 before relaxation), roughly an order-of-magnitude collapse (tests/test_sorting_regime.py::test_sorting_demixes_cohesively, tests/test_cpm_sorting.py::test_checkerboard_sorts_cohesively_in_composite, tests/test_cpm_sorting_spike.py::test_checkerboard_demixes_and_stays_cohesive) |
| sorting-clump-stays-cohesive | kind=observable path=obs.cell_pixels expr=abs(last(obs.cell_pixels) - first(obs.cell_pixels)) / first(obs.cell_pixels) | op < value 0.1 provenance cell_pixels ~1530 held to <1% drift over ~600 MCS (well inside the 10% cohesion bound) while hetero_frac collapsed -- the clump sorted, it did not dissolve (tests/test_sorting_regime.py::test_sorting_demixes_cohesively, tests/test_cpm_sorting.py::test_checkerboard_sorts_cohesively_in_composite) |
| sorting-conserves-cell-count | kind=observable path=obs.n_type1 expr=last(obs.n_type1) | op == value 32.0 provenance n_type1 == n_type2 == 32 for the full ~600 MCS run -- no cell created, destroyed, or merged, just moved (tests/test_cpm_sorting.py::test_checkerboard_sorts_cohesively_in_composite) |
| sorting-neutral-J-does-not-sort | kind=observable path=obs.hetero_frac expr=last(obs.hetero_frac) | op > value 0.5 provenance a neutral-J composite re-run (all J=8): neutral J (all=8), T=10, same checkerboard stays mixed at hetero_frac ~0.52 over 600 MCS vs ~0.06 for the differential-J regime -- the causality control for J. Documented regime sweep, not a committed pytest in this build. |
| condensate-phase-separates | kind=observable path=obs.phi_var expr=last(obs.phi_var) | op > value 0.3 provenance phi_var ~7e-5 -> ~0.38 over 10000 steps (64x64 field, M=1, kappa=0.5, dt=0.002) -- two domains formed, bounded by a sharp emergent interface (tests/test_sorting_regime.py::test_cahn_hilliard_phase_separates_mass_conserved, tests/test_cahn_hilliard.py::test_spinodal_decomposition_conserves_mass_and_stays_bounded) |
| condensate-conserves-mass | kind=observable path=obs.phi_mean expr=abs(last(obs.phi_mean) - first(obs.phi_mean)) | op < value 0.001 provenance phi_mean drift ~4e-18 over 10000 steps (mass-conserving by construction: the update is a divergence of a flux) (tests/test_sorting_regime.py::test_cahn_hilliard_phase_separates_mass_conserved, tests/test_cahn_hilliard.py::test_spinodal_decomposition_conserves_mass_and_stays_bounded) |
| condensate-stays-bounded | kind=observable path=obs.phi_min expr=last(obs.phi_min) | op > value -1.05 provenance phi bounded to [-0.989, 0.982] at run end (no NaN); the process raises loudly if dt exceeds the ~dx^4/(16*M*kappa) stability limit rather than emitting NaN (tests/test_cahn_hilliard.py::test_spinodal_decomposition_conserves_mass_and_stays_bounded, tests/test_cahn_hilliard.py::test_dt_above_stability_limit_raises_instead_of_nan) |


## Open decisions
- Four of the 9 planned spatial-counterpart studies are built and run so far — cell-environment-coupling-spatial (the flagship), cell-cell-coupling-spatial, disintegration-spatial, and growth-and-division-spatial. The remaining patterns — cellular-interface, molecular-interfaces, biomolecular-complementarity, autopoiesis, development-and-evolution — are named in the design spec's increment plan; some are planned, biomolecular-complementarity is in progress (code done, study.yaml held). Treat this investigation's verdict as scoped to the built studies only until the remaining increments land.
- Chemotaxis toward the sensed field (directed up-gradient motion) is explicitly deferred to a follow-up variant of the flagship composite; the current flagship cell does not chemotax, only senses, metabolizes, grows, and secretes.
